# Rag

## Utils

In [1]:
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_ollama import OllamaEmbeddings
from langchain_classic.chains.retrieval_qa.base import RetrievalQA
from huggingface_hub import login
import os
from dotenv import load_dotenv
import subprocess

/home/vignesh/.nlpvenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_22001/2625501338.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [2]:
from pathlib import Path
import logging
from tqdm import tqdm
# Tell pypdf to only print errors, which hides the font warnings
logging.getLogger("pypdf").setLevel(logging.ERROR)


from langchain_ollama import ChatOllama

OLLAMA_MODEL_NAME = 'deepseek-r1:latest'
RETRIVER_MODEL_NAME = 'llama3.2:latest'

def get_ollama_model(model_name,num_ctx=4046,num_predict=8046,keep_alive="2m",stream=False):
    return  ChatOllama(
            model=model_name,
            validate_model_on_init=True,
            num_ctx= num_ctx,
            num_predict=num_predict,
            stream=stream,
            keep_alive=keep_alive
        )

def stop_ollama(model:str):
    """
    Stop the Ollama model if it is running.
    """
    try:
        # Check if the model is running
        result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
        if model in result.stdout:
            # Stop the model
            subprocess.run(["ollama", "stop", model])
            print(f"Stopped Ollama model: {model}")
        else:
            print(f"Ollama model {model} is not running.")
    except Exception as e:
        print(f"Error stopping Ollama model: {e}")



def get_pdf_docs_via_text_spliter(folder_path:Path):
    file_paths = [os.path.join(folder_path,file) for file in os.listdir(folder_path)]
    docs = []
    text_spliter = RecursiveCharacterTextSplitter(chunk_size=2046,chunk_overlap=1046)
    print(f"Total no of files in {folder_path}: {len(os.listdir(folder_path))}")
    
    for path in tqdm(file_paths, desc="Processing PDFs", leave=True):
        PDF_loader = PyPDFLoader(file_path=str(path))
        pdf_doc = PDF_loader.load()
        docs.extend(text_spliter.split_documents(pdf_doc))
    
    return docs
    

In [3]:

class Vector_DB_store:
    def __init__(self,docs:list = None,model:str = 'nomic-embed-text:latest',base_url:str = 'http://127.0.0.1:11434',vector_db_path:Path = r'vector_db/Chroma/polymer_model_papers'):
        self._model = model
        self._docs = docs
        self._persist_path = vector_db_path
        os.makedirs(self._persist_path,exist_ok=True)
        self._embed = OllamaEmbeddings(model=self._model,base_url=base_url)
        
    def save(self,batch_size=4):
        if self._docs ==None:
            raise ValueError(f"Error: docs must me passed to save, docs:{self._docs}")
        vector_db = self.load()
                
        for i in tqdm(range(0,len(self._docs),batch_size), desc="Generating Embeddings & Storing"):
                    batch = self._docs[i : i + batch_size]
                    vector_db.add_documents(batch)
        # Chroma.from_documents(
        #     self._docs,
        #     embedding=self._embed,
        #     persist_directory=self._persist_path,
        # )
        stop_ollama(self._model)
    
    def load(self,):
        return Chroma(
            persist_directory=self._persist_path,
            embedding_function=self._embed,
        )
        
    def get_retriver(self,k:int=4):
        vector_db = self.load()
        return vector_db.as_retriever(k=k)

In [4]:
template = """You are a helpful assistant that can answer questions and return the reponse in markdown format.
            You are also provided with latest and new context relavent to query. Use the following pieces of context to add more information and answer the question at the end.\n\n{context}\n\nQuestion: {question}\nHelpful Answer:"""
prompt = PromptTemplate.from_template(template)

## Basic (Standard)

In [1]:
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_ollama import OllamaEmbeddings
from langchain_classic.chains.retrieval_qa.base import RetrievalQA
from huggingface_hub import login
import os
from dotenv import load_dotenv
import subprocess

/home/vignesh/.nlpvenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_21273/2625501338.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


### Stop llm (ollama) and Read PDF Docs

In [2]:
from pathlib import Path
import logging
from tqdm import tqdm
# Tell pypdf to only print errors, which hides the font warnings
logging.getLogger("pypdf").setLevel(logging.ERROR)
def stop_ollama(model:str):
    """
    Stop the Ollama model if it is running.
    """
    try:
        # Check if the model is running
        result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
        if model in result.stdout:
            # Stop the model
            subprocess.run(["ollama", "stop", model])
            print(f"Stopped Ollama model: {model}")
        else:
            print(f"Ollama model {model} is not running.")
    except Exception as e:
        print(f"Error stopping Ollama model: {e}")



def get_pdf_docs_via_text_spliter(folder_path:Path):
    file_paths = [os.path.join(folder_path,file) for file in os.listdir(folder_path)]
    docs = []
    text_spliter = RecursiveCharacterTextSplitter(chunk_size=2046,chunk_overlap=1046)
    print(f"Total no of files in {folder_path}: {len(os.listdir(folder_path))}")
    
    for path in tqdm(file_paths, desc="Processing PDFs", leave=True):
        PDF_loader = PyPDFLoader(file_path=str(path))
        pdf_doc = PDF_loader.load()
        docs.extend(text_spliter.split_documents(pdf_doc))
    
    return docs
    

In [6]:
PAPERS = 'inputs/pdf'
docs = get_pdf_docs_via_text_spliter(PAPERS)


Total no of files in inputs/pdf: 20


Processing PDFs: 100%|██████████| 20/20 [00:42<00:00,  2.15s/it]


In [7]:
print(docs[0])

page_content='Quantitative Structure�Permittivity Relationship Study of a Series
of Polymers
Published as part of ACS Materials Au virtual special issue “2023 Rising Stars”.
Yevhenii Zhuravskyi, Kweeni Iduoku, Meade E. Erickson, Anas Karuth, Durbek Usmanov,
Gerardo Casanola-Martin, Maqsud N. Sayfiyev, Dilshod A. Ziyaev, Zulayho Smanova,
Alicja Mikolajczyk,* and Bakhtiyor Rasulev*
Cite This: ACS Mater. Au 2024, 4, 195−203
 Read Online
ACCESS
 Metrics & More
 Article Recommendations
ABSTRACT: Dielectric constant is an important property which
is widely utilized in many scientific fields and characterizes the
degree of polarization of substances under the external electric
field. In this work, a structure−property relationship of the
dielectric constants (ε) for a diverse set of polymers was
investigated. A transparent mechanistic model was developed
withtheapplicationofamachinelearningapproachthatcombines
geneticalgorithmandmultiplelinearregressionanalysis,toobtain
amechanisticallyexplai

In [8]:
print(type(docs))

<class 'list'>


### Vector DB (Chroma)

In [35]:

class Vector_DB_store:
    def __init__(self,docs:list = None,model:str = 'nomic-embed-text:latest',base_url:str = 'http://127.0.0.1:11434',vector_db_path:Path = r'vector_db/Chroma/polymer_model_papers'):
        self._model = model
        self._docs = docs
        self._persist_path = vector_db_path
        os.makedirs(self._persist_path,exist_ok=True)
        self._embed = OllamaEmbeddings(model=self._model,base_url=base_url)
        
    def save(self,batch_size=4):
        if self._docs ==None:
            raise ValueError(f"Error: docs must me passed to save, docs:{self._docs}")
        vector_db = self.load()
                
        for i in tqdm(range(0,len(self._docs),batch_size), desc="Generating Embeddings & Storing"):
                    batch = self._docs[i : i + batch_size]
                    vector_db.add_documents(batch)
        # Chroma.from_documents(
        #     self._docs,
        #     embedding=self._embed,
        #     persist_directory=self._persist_path,
        # )
        stop_ollama(self._model)
    
    def load(self,):
        return Chroma(
            persist_directory=self._persist_path,
            embedding_function=self._embed,
        )
        
    def get_retriver(self,k:int=4):
        vector_db = self.load()
        return vector_db.as_retriever(k=k)

In [37]:
# vector_db_store = Vector_DB_store(docs=docs)
vector_db_store = Vector_DB_store()

#### Store Vector DB

In [ ]:
vector_db_store.save()

In [38]:
retriver = vector_db_store.get_retriver(k=7)

/tmp/ipykernel_21273/891149308.py:25: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  return Chroma(


### Ollama Chat model (DeepSeek-r1)

In [4]:
from langchain_ollama import ChatOllama

In [66]:

OLLAMA_MODEL_NAME = 'deepseek-r1:latest'

ollama_model = ChatOllama(
    model=OLLAMA_MODEL_NAME,
    validate_model_on_init=True,
    num_ctx= 22921,
    num_predict=8046,
    stream=True
)

In [16]:
system_prompt = "You are a helpful assistant that can answer questions and return the reponse in markdown format."
message = [
    ("system",system_prompt),
    ("human","explain how vllm works, and how different from ollama and loading and using via hugging face transformers?")
]
for chunk in ollama_model.stream(message):
    print(chunk.text,end="", flush=True)


Here's a breakdown of how **vLLM**, **Ollama**, and **Hugging Face Transformers** work and their key differences:

---

### 🔍 What is vLLM?
- **Purpose**: A highly optimized inference engine for large language models (LLMs), focusing on performance, scalability, and memory efficiency.
- **Core Features**:
  - Uses asynchronous request processing to handle multiple inputs concurrently.
  - Implements "PagedAttention" to dynamically manage GPU memory during long-sequence generation, reducing wasted space.
  - Supports high-throughput serving via an API (similar to OpenAI's GPT API).
- **How it Works**:  
  1. Loads a model from storage or a framework like PyTorch/TensorRT-LLM.  
  2. Processes requests in batches using GPU parallelism and streaming token generation.  
  3. Optimizes memory allocation by breaking attention computations into smaller chunks ("pages").  

**Usage**:  
```bash
# Example command to serve a model via API (e.g., Llama 2)
vllm server --model meta-llama/Llama-2-7

In [17]:
# stop_ollama('nomic-embed-text:latest')
stop_ollama(OLLAMA_MODEL_NAME)

Stopped Ollama model: deepseek-r1:latest


⠙ 

In [18]:
query = 'Explain About TG glass transition temperature in polymers'
message_2 = [
    {'role':'system', 'content' : system_prompt},
    {'role':'user', 'content':query}
]

for chunk in ollama_model.stream(message_2):
    print(chunk.text,end="",flush=True)


Okay, let's break down the concept of **Tg (Glass Transition Temperature)** in polymers.

## What is Glass Transition?

Imagine heating a solid polymer slowly. At first, nothing much happens until you reach its melting point (*Tm*), where it turns into a liquid melt. However, before reaching *Tm*, there's a crucial change: the material transforms from a **glassy (solid-like)** state to a **rubbery or viscous (soft solid/liquid-like)** state.

This transition isn't sharp like melting; instead, it's more of a gradual softening process as temperature increases. The point where this significant change in physical properties occurs is defined by the **Glass Transition Temperature (*Tg*)**.

### Definition

*   ***Tg*** is the specific temperature at which an amorphous or semi-crystalline polymer transitions from being relatively hard and brittle (glassy state) to soft, viscous, and flexible (rubbery/vitreous liquid state).
*   It marks a change in the polymer's physical properties rather t

In [19]:
stop_ollama(OLLAMA_MODEL_NAME)

Stopped Ollama model: deepseek-r1:latest


In [55]:
template = """You are a helpful assistant that can answer questions and return the reponse in markdown format.
            You are also provided with latest and new context relavent to query. Use the following pieces of context to add more information and answer the question at the end.\n\n{context}\n\nQuestion: {question}\nHelpful Answer:"""
prompt = PromptTemplate.from_template(template)

### RAG Pipeline

In [27]:
qa_chain = RetrievalQA.from_llm(
    llm=ollama_model,
    retriever=retriver,
    prompt=prompt,
    # return_source_documents=True,
    )

In [24]:
for chunk in qa_chain.stream({"query":"what is Tg target in polymer properties, what are the latest advancment in predicting Tg target using Deep Learning, give the model preformance"}):
  print(chunk['result'],end="",flush=True)


Okay, here's a breakdown of the provided text regarding **Tg** (glass transition temperature) as a polymer property target and recent advances using Deep Learning:

## What is Tg Target?

*   **Tg** stands for **Glass Transition Temperature**, which is a key physical property in polymers. It represents the temperature at which an amorphous or semicrystalline polymer transitions from a glassy, rigid state to a rubbery (viscoelastic) state.
*   As a target "in polymer properties," Tg refers to the specific **goal** for designing and developing polymers that can function reliably in high-temperature environments. Polymers with higher Tg values are sought after for applications requiring thermal stability above certain temperatures.

## Latest Advances in Predicting Tg using Deep Learning

*   The context discusses combining unique aspects into a single machine learning framework, including feature space development based on topological descriptors and integration of unsupervised and supe

In [26]:
stop_ollama(model=OLLAMA_MODEL_NAME) 
stop_ollama('nomic-embed-text:latest')

⠙ 

Stopped Ollama model: deepseek-r1:latest


⠙ ⠸ ⠸ ⠼ ⠴ ⠧ 

Stopped Ollama model: nomic-embed-text:latest


⠇ ⠇ 

In [27]:
for chunk in qa_chain.stream({"query":"what is Tg target in polymer properties, what are the latest advancment in predicting Tg target using Deep Learning, give the model preformance"}):
  print(chunk['result'],end="",flush=True)


Okay, let's break down your question based on the provided text:

**1. What is Tg target in polymer properties?**

Tg refers to **Glass Transition Temperature**, a critical property of polymers.

*   It represents the temperature at which a solid polymer transitions from a hard, glassy state (solid-like behavior) into a rubbery, viscous liquid state.
*   At temperatures below Tg, the polymer is brittle and rigid. Above Tg, it becomes softer, more flexible, and viscous rather than elastic like a liquid.
*   **Significance:** It's crucial for determining if a polymer can be used above its glass transition temperature without melting (softening point) or becoming too soft/flowable.

**2. What are the latest advancements in predicting Tg using Deep Learning?**

The context provided focuses on recent work, specifically from a paper published in *ijms* journal (likely 2023 given the PMID prefix). Key points:

*   **Approach:** The advancement involves combining unique aspects into an ML fra

In [28]:
stop_ollama(model=OLLAMA_MODEL_NAME) 
stop_ollama('nomic-embed-text:latest')

⠋ 

Stopped Ollama model: deepseek-r1:latest


⠙ ⠸ ⠸ ⠼ ⠦ ⠦ 

Stopped Ollama model: nomic-embed-text:latest


⠧ 

In [29]:
for chunk in qa_chain.stream({"query":"what is Tg target in polymer properties, what are the latest advancment in predicting Tg target using Deep Learning, give the model preformance"}):
  print(chunk['result'],end="",flush=True)


Okay, based on the provided text, here's a breakdown of Tg and related information:

1.  **What is Tg target in polymer properties?**
    *   From the context, "Tg" refers to the **Glass Transition Temperature (Tg)** of polymers. It represents the temperature at which an amorphous or non-crystalline polymer transitions from a rigid, glassy state to a soft, rubbery state as it is heated.
    *   The text implies that Tg is often used as a target property for designing and developing **sustainable high-Tg polymers** (polymers with targeted high glass transition temperatures), especially in the context of "high-Tg sustainable polymer chemistry" mentioned several times. This suggests focusing on achieving specific, relatively high Tg values using environmentally friendly or novel approaches.

2.  **Latest advancements in predicting Tg target using Deep Learning:**
    *   The text mentions combining unique aspects (like topological descriptors and integrating unsupervised/supervised learn

In [30]:
stop_ollama(model=OLLAMA_MODEL_NAME) 
stop_ollama('nomic-embed-text:latest')

Stopped Ollama model: deepseek-r1:latest
Stopped Ollama model: nomic-embed-text:latest


⠙ 

In [16]:
for chunk in qa_chain.stream({"query":"what is Tg target in polymer properties, what are the latest advancment in predicting Tg target using Deep Learning, give the model preformance"}):
  print(chunk['result'],end="",flush=True)


Okay, let's break down this query based on the provided context:

**Tg Target in Polymer Properties:**

Glass transition temperature (Tg) refers to a specific **target property** related to polymer behavior. It represents the glass transition temperature.

*   **What it is:** Tg is the temperature at which an amorphous or semicrystalline polymer transitions from a hard, glassy state into a rubbery liquid state as it absorbs heat and becomes more flexible.
*   **Why it's important (target):** In polymer science, knowing Tg *a priori* allows engineers to:
    *   Predict the use temperature limits of a polymer.
    *   Select appropriate polymers for specific applications requiring certain mechanical properties at elevated temperatures.
    *   Design polymeric materials with desired thermal stability or flexibility characteristics.

**Latest Advancements in Deep Learning (DL) Tg Prediction:**

The provided context indicates that DL techniques have been applied to predict Tg, often buil

In [19]:
stop_ollama(model=OLLAMA_MODEL_NAME) 
stop_ollama('nomic-embed-text:latest')

⠙ ⠙ 

Stopped Ollama model: deepseek-r1:latest


⠹ ⠸ ⠼ ⠼ ⠦ ⠦ 

Stopped Ollama model: nomic-embed-text:latest


⠧ ⠏ 

In [22]:
for chunk in qa_chain.stream({"query":"what is Tg target in polymer properties, what are the latest advancment in predicting Tg target using Deep Learning, give the model preformance"}):
  print(chunk['result'],end="",flush=True)


Okay, here's the helpful answer based on the provided text:

**Glass Transition Temperature (Tg)**

*   **What is Tg Target?**
    The glass transition temperature (Tg) is a key physical property of polymers. It represents the temperature below which the polymer transitions from a hard and relatively rigid "glassy" state to a softer, rubbery or viscous "leathery" state under an applied load.

**Latest Advancements in Tg Prediction using Deep Learning**

Several recent advancements have been mentioned in the context provided:
1.  **Machine Learning Integration:** There has been significant work combining unique aspects like topological descriptors and integrating unsupervised (e.g., Random Forest for feature importance) and supervised learning approaches.
2.  **Experimental Validation:** Notably, a paper combined machine learning predictions with *experimental validation* of new high-Tg sustainable polymers to verify the model's accuracy.

**Model Performance**

The provided context de

In [25]:
stop_ollama(model=OLLAMA_MODEL_NAME) 
stop_ollama('nomic-embed-text:latest')

Stopped Ollama model: deepseek-r1:latest
Stopped Ollama model: nomic-embed-text:latest


In [30]:
for chunk in qa_chain.stream({"query":"what is Tg target in polymer properties, what are the latest advancment in predicting Tg target using Deep Learning, give the model preformance"}):
  print(chunk['result'],end="",flush=True)


Okay, let's break down the query into its components:

1.  **What is Tg target in polymer properties?**
    *   `Tg` stands for **glass transition temperature** (Tg).
    *   As a target property (`target` or `Tg_target`), it refers to the specific threshold temperature, usually measured in degrees Celsius (°C) or Kelvin (K), at which an amorphous polymer transitions from a hard, glassy state into a rubbery or viscous liquid-like state. It's crucial for determining how well a polymer can withstand heat without becoming too soft or flowing.

2.  **What is the latest advancement in predicting Tg target using Deep Learning?**
    *   The provided context doesn't detail groundbreaking new DL models beyond the specific paper it seems to summarize, but describes their work as an example of recent progress.
    *   It highlights that existing methods often rely on QSPR (Quantitative Structure-Property Relationship) models built with small or congeneric datasets using older techniques like ML

In [31]:
stop_ollama(model=OLLAMA_MODEL_NAME) 
stop_ollama('nomic-embed-text:latest')

Stopped Ollama model: deepseek-r1:latest
Stopped Ollama model: nomic-embed-text:latest


## Quesry Translation

In [26]:
OLLAMA_MODEL_2 = 'llama3.2:latest'

ollama_model_2 = ChatOllama(
    model=OLLAMA_MODEL_2,
    validate_model_on_init=True,
    num_ctx=2046,
    num_predict=6046,
    keep_alive=0
)

### Utils

In [6]:
from langchain_core.output_parsers import StrOutputParser

TEMPLATE = """You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines.Note:only give qestions noe extra text. Original question: {question}"""
def utils_get_queries(question:str,ollama_model:ChatOllama = get_ollama_model(model_name='llama3.2:latest'), template:str= TEMPLATE):
    prompt_perspectives = PromptTemplate.from_template(template)
    
    generate_queries = (
        prompt_perspectives | 
        ollama_model |
        StrOutputParser() | 
        (lambda x : x.split("\n"))
    )
    queries_generatied = generate_queries.invoke({"question":question})
    queries_generatied = [querie for querie in queries_generatied if querie and querie.strip()]
    # result = queries_generatied.append(question)
    return queries_generatied

### Multi Query

In [86]:
template = """You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines.Note:only give qestions noe extra text. Original question: {question}"""
prompt_perspectives = PromptTemplate.from_template(template)


In [87]:
from langchain_core.output_parsers import StrOutputParser

In [88]:
generate_queries = (
    prompt_perspectives | 
    ollama_model_2 |
    StrOutputParser() | 
    (lambda x : x.split("\n"))
)

In [89]:
queries_generatied = generate_queries.invoke({"question":"what is Tg target in polymer properties, what are the latest advancment in predicting Tg target using Deep Learning, give the model preformance"})

In [93]:
print(queries_generatied)

['What are the current polymer properties that have a Tg target?', 'How does Tg target relate to the overall performance of polymers in various applications?', 'What are some recent advances in predicting Tg target using machine learning algorithms?', '', 'How do deep learning models perform in predicting the Tg target for different types of polymers?', 'Can any specific features or characteristics of polymers be used as input to predict their Tg target accurately?', 'Are there any limitations or challenges associated with using deep learning for Tg target prediction?', '', 'What are some potential applications of accurately predicted Tg targets in polymer engineering and materials science?', 'How can the performance of deep learning models for Tg target prediction be evaluated and compared across different datasets?', 'Can combining multiple machine learning algorithms improve the accuracy of Tg target predictions?', 'what is Tg target in polymer properties, what are the latest advanc

In [42]:
from langchain_core.load import dumps, loads

In [119]:

def get_queries(question):
    generate_queries = (
        prompt_perspectives | 
        ollama_model_2 |
        StrOutputParser() | 
        (lambda x : x.split("\n"))
    )
    queries_generatied = generate_queries.invoke({"question":question})
    queries_generatied = [querie for querie in queries_generatied if querie]
    # result = queries_generatied.append(question)
    return queries_generatied

def get_unique_union(documents: list[list]):
    """ Unique union of retrieved docs """
    # Flatten list of lists, and convert each Document to string
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # Get unique documents
    unique_docs = list(set(flattened_docs))
    # Return
    return [loads(doc) for doc in unique_docs]


In [120]:
question = "what is Tg target in polymer properties, what are the latest advancment in predicting Tg target using Deep Learning, give the model preformance"


In [121]:
queries = get_queries(question)
queries

['What is the TG target in polymer properties and its significance?',
 'How does the TG target relate to the thermal stability of polymers?',
 'What are the current advancements in predicting the TG target of polymers using machine learning algorithms?',
 'Can deep learning models accurately predict the TG target of polymers based on their molecular structure or other properties?',
 'What is the performance of a recent deep learning model for predicting TG targets in polymers, and how does it compare to traditional methods?']

In [122]:
retrival_chain = get_queries | retriver.map() | get_unique_union

In [123]:
question = "what is Tg target in polymer properties, what are the latest advancment in predicting Tg target using Deep Learning, give the model preformance"

In [124]:
retriver_docs = retriver.invoke(question)

In [125]:
len(retriver_docs)

4

In [ ]:

retrived_docs = retrival_chain.invoke({"question":question})
len(retrived_docs)

In [126]:

retrived_docs = retrival_chain.invoke({"question":question})
len(retrived_docs)

/tmp/ipykernel_21273/1821810733.py:20: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


20

In [127]:
stop_ollama("nomic-embed-text:latest")

Stopped Ollama model: nomic-embed-text:latest


In [128]:
stop_ollama(OLLAMA_MODEL_2)

Stopped Ollama model: llama3.2:latest


In [129]:
from operator import itemgetter

In [130]:
multi_query_rg = (
    {
        "context": retrival_chain,
        "question": itemgetter("question")
    }
    | prompt
    | ollama_model
    | StrOutputParser()
)

In [131]:

print(multi_query_rg.invoke({"question":question}))

/tmp/ipykernel_21273/1821810733.py:20: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]



## What is Tg (Glass Transition Temperature) in Polymer Properties?

Tg stands for Glass Transition Temperature. It marks the approximate temperature below which a polymer transitions from a relatively soft, flexible, rubbery state to a hard, brittle, glass-like state.

### Key Characteristics:

*   **Thermal Stability:** Polymers with higher Tg values exhibit greater thermal stability and maintain their mechanical integrity at elevated temperatures.
*   **Importance:** It's a critical indicator for many applications (e.g., aerospace, energy, biomedical, electronics) where materials must withstand challenging thermal environments. Most common biopolymers have lower Tg ranges (~55-75°C).
*   **Factors:** Theoretically dependent on chain mobility or free volume, which is influenced by molecular weight, cross-links, side groups, and chain ends.

---

## Latest Advancements in Predicting Polymer Tg using Deep Learning (from the provided context)

Recent advancements leverage machine learn

In [132]:
embedding_model = "nomic-embed-text:latest"
stop_ollama(OLLAMA_MODEL_NAME)
stop_ollama(OLLAMA_MODEL_2)
stop_ollama(embedding_model)

Stopped Ollama model: deepseek-r1:latest


⠙ 

Stopped Ollama model: llama3.2:latest
Stopped Ollama model: nomic-embed-text:latest


### RAG Fusion

In [26]:
question = "what is Tg target in polymer properties, what are the latest advancment in predicting Tg target using Deep Learning, give the model preformance"
query_model = 'llama3.2:latest'
embedding_model = "nomic-embed-text:latest"
queries = utils_get_queries(ollama_model=get_ollama_model(model_name=query_model),question=question)

In [27]:
stop_ollama(query_model)

Stopped Ollama model: llama3.2:latest


In [28]:
print(queries)
print(len(queries))

['What are some key characteristics of Tg targets in polymers that predict their performance?', 'How does machine learning approach impact predictions on Tg targets in advanced materials?', 'Can deep learning models be optimized for precise Tg target prediction and what techniques would improve model performance?', 'What are the most recent advancements in deep learning models predicting Tg targets and how do these relate to polymer properties?', 'How do different deep learning architectures impact the accuracy of Tg target predictions, with specific focus on polymer systems?', 'Can the performance of a deep learning model for Tg target prediction be enhanced by incorporating physical and chemical properties of polymers?', 'What is the current state of knowledge in predicting Tg targets using machine learning models, including any recent breakthroughs or challenges encountered?', 'How do Tg targets relate to the overall performance of polymer materials, with specific consideration for 

In [29]:
from langchain_core.load import dumps, loads

def reciprocal_rank_fusion(results: list[list], k=60):
    """ Reciprocal_rank_fusion that takes multiple lists of ranked documents 
        and an optional parameter k used in the RRF formula """
    
    # Initialize a dictionary to hold fused scores for each unique document
    fused_scores = {}

    # Iterate through each list of ranked documents
    for docs in results:
        # Iterate through each document in the list, with its rank (position in the list)
        for rank, doc in enumerate(docs):
            # Convert the document to a string format to use as a key (assumes documents can be serialized to JSON)
            doc_str = dumps(doc)
            # If the document is not yet in the fused_scores dictionary, add it with an initial score of 0
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            # Retrieve the current score of the document, if any
            previous_score = fused_scores[doc_str]
            # Update the score of the document using the RRF formula: 1 / (rank + k)
            fused_scores[doc_str] += 1 / (rank + k)

    # Sort the documents based on their fused scores in descending order to get the final reranked results
    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

    # Return the reranked results as a list of tuples, each containing the document and its fused score
    return reranked_results


In [30]:
vector_db = Vector_DB_store()

fusion_retriver = vector_db.get_retriver(k=3)

In [31]:
retrival_chain_fusion = utils_get_queries | fusion_retriver.map() | reciprocal_rank_fusion

In [32]:
docs = retrival_chain_fusion.invoke({"question": question})
len(docs)


/tmp/ipykernel_9244/1429506535.py:26: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


20

In [ ]:
# stop_ollama(model=embedding_model)
# stop_ollama(model=query_model)

Stopped Ollama model: nomic-embed-text:latest
Stopped Ollama model: llama3.2:latest


In [50]:
DEEPSEEK_MODEL = "deepseek-r1:latest"
fusion_ollama_model = get_ollama_model(model_name=DEEPSEEK_MODEL,num_ctx=22921)

In [51]:
from operator import itemgetter

fusion_query_rag = (
    {
        "context": retrival_chain_fusion,
        "question": itemgetter("question")
    }
    | prompt
    | fusion_ollama_model
    | StrOutputParser()
)

In [52]:
output = fusion_query_rag.invoke({"question":question})

/tmp/ipykernel_9244/1429506535.py:26: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


In [53]:
print(output)


Here is a breakdown based on the provided context:

**1. What is Tg target in polymer properties?**

Glass Transition Temperature (Tg) refers to the approximate temperature below which a macromolecular system changes its state from relatively soft, flexible, or rubbery to hard, brittle, and glass-like [2].

*Tg values are critical indicators of a polymer's thermal properties.* They mark the point where a polymer transitions from a rigid, glassy state to a softer, rubbery state. Polymers with higher Tg values exhibit enhanced thermal stability, allowing them to maintain mechanical integrity even under elevated temperatures.

**2. What are the latest advancements in predicting Tg target using Deep Learning?**

Several recent studies have employed machine learning (ML), particularly deep learning (DL) techniques, to predict polymer glass transition temperatures:

*   **Chemical Language Processing (SMILES-based):** Guang Chen et al. proposed a model that uses Recurrent Neural Networks (R

In [54]:
stop_ollama(DEEPSEEK_MODEL)
stop_ollama(query_model)
stop_ollama(embedding_model)

Stopped Ollama model: deepseek-r1:latest


⠋ ⠙ 

Stopped Ollama model: llama3.2:latest
Stopped Ollama model: nomic-embed-text:latest


⠹ 

### Decomposition

In [22]:
template = """You are a helpful assistant that generates multiple sub-questions related to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that can be answers in isolation. \n
Generate multiple search queries related to: {question} \n
.Note:only give questions, no extra text.\n
Output (3 queries):"""
prompt_decomposition = PromptTemplate.from_template(template)

In [23]:
retriver_llm = get_ollama_model(model_name=RETRIVER_MODEL_NAME)

In [36]:
generate_queries_decomposition = (prompt_decomposition | retriver_llm | StrOutputParser()| (lambda x: x.split("\n")))

In [37]:
question = "what is Tg target in polymer properties, what are the latest advancment in predicting Tg target using Deep Learning, give the model preformance"

queries_generatied = generate_queries_decomposition.invoke({"question":question})

In [38]:
queries_generatied

['1. What is the significance of Tg (glass transition temperature) in determining polymer properties and behavior?',
 '2. How do current machine learning algorithms improve prediction accuracy for Tg values in polymers using deep learning techniques?',
 '3. What are some recent advancements in developing and training neural networks to accurately predict Tg targets in various types of polymeric materials?']

In [39]:
template = """Here is the question you need to answer:

\n --- \n {question} \n --- \n

Here is any available background question + answer pairs:

\n --- \n {q_a_pairs} \n --- \n

Here is additional context relevant to the question: 

\n --- \n {context} \n --- \n

Use the above context and any background question + answer pairs to answer the question: \n {question}
"""

decomposition_prompt = PromptTemplate.from_template(template)


In [47]:
vector_db_store = Vector_DB_store()
retriever = vector_db_store.get_retriver(k=7)

In [49]:
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

def format_qa_pair(question, answer):
    """Format Q and A pair"""
    
    formatted_string = ""
    formatted_string += f"Question: {question}\nAnswer: {answer}\n\n"
    return formatted_string.strip()

# llm
llm = get_ollama_model(model_name=OLLAMA_MODEL_NAME,num_ctx=6069)

q_a_pairs = ""
for q in queries_generatied:
    
    rag_chain = (
    {"context": itemgetter("question") | retriever, 
     "question": itemgetter("question"),
     "q_a_pairs": itemgetter("q_a_pairs")} 
    | decomposition_prompt
    | llm
    | StrOutputParser())

    answer = rag_chain.invoke({"question":q,"q_a_pairs":q_a_pairs})
    q_a_pair = format_qa_pair(q,answer)
    q_a_pairs = q_a_pairs + "\n---\n"+  q_a_pair


In [51]:
print(answer)


Here is an answer to your question:


### Recent Advancements in Neural Network Predictions for Polymer Tg

Recent advancements in neural network-based predictions of glass transition temperatures (Tg) have significantly improved accuracy and applicability. These developments leverage deep learning techniques such as **transformer models** and **recurrent neural networks (RNNs)** to process raw chemical structure data directly, without relying on traditional molecular descriptors or fingerprints.

1. **Direct SMILES Processing**: Models now use polymer repeat unit structures encoded in SMILES strings as sequential inputs for RNNs or Transformers. This approach eliminates the need for generating additional structural parameters and enhances computational efficiency by leveraging end-to-end learning from chemical syntax alone.

2. **Hybrid Feature Engineering**: Scientific insights, such as identifying backbone index (based on steric hindrance) and rotational degrees of freedom, are int

## Rounting

# semantic Routing

In [17]:
from langchain_community.utils.math import cosine_similarity
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

In [6]:
# Two prompts
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{query}"""

math_template = """You are a very good mathematician. You are great at answering math questions. \
You are so good because you are able to break down hard problems into their component parts, \
answer the component parts, and then put them together to answer the broader question.

Here is a question:
{query}"""


In [7]:
embedding_model = "nomic-embed-text:latest"
base_url = 'http://127.0.0.1:11434'
embedder = OllamaEmbeddings(model=embedding_model,base_url=base_url)


In [8]:
prompt_templates = [physics_template,math_template]
prompt_embeddes = embedder.embed_documents(prompt_templates)
print(prompt_embeddes)

[[0.038353715, 0.08012494, -0.14337745, -0.019404389, 0.08627023, 0.034016073, 0.040518135, 0.010563064, 0.007001047, -0.024331179, 0.0015425527, 0.017804287, 0.042052504, 0.007821103, -0.0344756, -0.03783338, 0.047883634, -0.12959132, 0.032205675, 0.012894707, -0.0717537, -0.041941732, -0.042224407, -0.018879898, 0.10378716, -0.028598383, -0.10100625, -0.025113178, -0.0042227083, -0.04359907, 0.0073161605, -0.020715753, -0.0081583755, 0.015352621, -0.07838562, -0.03359878, 0.024930036, 0.023953319, -0.002097443, 0.029541897, 0.02598957, -0.05888821, -0.006738316, -0.074946195, 0.08210979, -0.0079086125, 0.011616006, -0.05866863, -0.0036017057, -0.0678093, -0.046651483, -0.048520375, 0.02252338, 0.02315494, 0.003534879, 0.039976433, 0.024936704, 0.03609188, 0.09150879, 0.032689877, 0.062736265, 0.028050333, 0.010782357, 0.045007005, 0.0044971416, 0.007674403, -0.027245691, 0.073478624, -0.027761418, -0.026694389, 0.037478603, -0.031545557, 0.039961334, 0.015185296, -0.011682765, 0.0423

In [15]:
query = "What is General Relativity?"
query_embeddings = embedder.embed_documents([query])
# print(query_embeddings)
similarity = cosine_similarity(query_embeddings,prompt_embeddes)
prompt_templates[similarity.argmax()]

"You are a very smart physics professor. You are great at answering questions about physics in a concise and easy to understand manner. When you don't know the answer to a question you admit that you don't know.\n\nHere is a question:\n{query}"

In [16]:
def prompt_router(input):
    query_embeddings = embedder.embed_documents(input['query'])
    similarity = cosine_similarity(query_embeddings,prompt_embeddes)
    most_similar = prompt_templates[similarity.argmax()]
    
    return PromptTemplate.from_template(most_similar)

In [18]:
ollama_llm = get_ollama_model(model_name="qwen3:8b")

In [22]:
chain = (
    {"query":RunnablePassthrough()}
     |RunnableLambda(prompt_router)
     |ollama_llm
     |StrOutputParser()
)

In [23]:
response = chain.invoke(query)

In [24]:
print(response)

General Relativity (GR) is Einstein's theory of gravity, published in 1915. It redefines gravity not as a force (as in Newtonian physics) but as the curvature of spacetime caused by mass and energy. Massive objects like stars and planets warp the fabric of spacetime, and this curvature dictates how objects move. For example, Earth orbits the Sun because it follows the "straightest possible path" in the Sun's warped spacetime. Key concepts include:  
- **Equivalence Principle**: Gravity and acceleration are locally indistinguishable.  
- **Spacetime Geometry**: Mass curves spacetime, and objects move along geodesics (the shortest paths in curved geometry).  
- **Predictions**: Gravitational time dilation, black holes, gravitational waves, and light bending near massive objects.  

GR has been confirmed by experiments like the bending of starlight, gravitational lensing, and the detection of gravitational waves. It remains the most accurate description of gravity in extreme conditions (e

In [25]:
math_query = "why infinite serires of 9's is equal to -1. is the infintes behave different compared to definet numbers?"
response = chain.invoke(math_query)
print(response)

The claim that the infinite series of 9's equals -1 arises from a specific mathematical context, not the standard real number system. Here's a breakdown:

### 1. **Understanding the Series**
   - The series in question is **not** $ 0.999\ldots $ (which equals 1 in real numbers). Instead, it refers to a **divergent series** like $ 9 + 99 + 999 + \ldots $, where each term grows exponentially.
   - In the **real number system**, this series **diverges** to infinity because the terms do not approach zero.

### 2. **10-Adic Numbers: A Different Number System**
   - In **10-adic numbers**, a non-standard number system, infinite series can converge even if they diverge in real numbers. The 10-adic numbers are defined such that the "digits" extend infinitely to the **left**, not the right.
   - The infinite sequence of 9's $ \ldots999 $ (infinite 9's to the left) is interpreted as a 10-adic number. Adding 1 to this number yields:
     $$
     \ldots999 + 1 = \ldots000 = 0 \quad \text{(in 10-ad